In [5]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.abstract_event_listener import AbstractEventListener
from selenium.webdriver.support.events import EventFiringWebDriver, AbstractEventListener
from selenium.webdriver import ActionChains
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import UnexpectedAlertPresentException
from selenium import webdriver
# from webdriver_auto_update.chrome_app_utils import ChromeAppUtils
# from webdriver_auto_update.webdriver_manager import WebDriverManager
import traceback

#* required
import base64
import re
import os #* ไม่ได้
import win32print
import win32api

#* setup
def setup_chrome():
    options = Options()
    options.add_experimental_option("debuggerAddress", "localhost:8989")
    driver = webdriver.Chrome(service=Service(r'C:\bin\chromedriver.exe'), options=options)
    return driver

def get_tabs():
    global merged_dict
    try:
        # if parent.winfo_exists():
        if True:
            print("รายงานจำนวนtabs")

            # * เก็บชื่อ title และ value ของ tab ที่เปิดอยู่
            title_list = []
            # title_list_Idx = [] #!เหมือนจะไม่ได้ใช้
            value_list = []
            # title_dict = {} #!เหมือนจะไม่ได้ใช้
            for idx, handle in enumerate(driver.window_handles):
                driver.switch_to.window(handle)
                # title_list_Idx.append(
                #     driver.title + "["+str(idx)+"]") #!เหมือนจะไม่ได้ใช้
                title_list.append(driver.title)

                value_list.append(driver.current_window_handle)
                # title_dict.update(
                #     {driver.title: driver.current_window_handle}) #!เหมือนจะไม่ได้ใช้

            # * เอาtitle มาทำให้ unique เพราะ title จะสามารถที่จะซ้ำกันได้
            unique_titles = []
            counter = {}
            for item in title_list:
                if item in counter:
                    counter[item] += 1
                    print("counter[item] คือไร: ", counter[item])
                    unique_titles.append(
                        f"{item}{counter[item]-1}")
                else:
                    counter[item] = 1
                    unique_titles.append(item)

            # * เอาList มารวมกัน
            merged_dict = dict(zip(unique_titles, value_list))
            print("มี tabs ไรบ้าง", merged_dict)
            
    except Exception as e:
        traceback_str = traceback.format_exc()
        print(f"An error occirred: {e}")
        print(traceback_str)
        
def fill_items(array_items=[]):
    sku_input_xpath = '/html/body/div[1]/div[2]/div[2]/div[2]/div[1]/div[1]/from/div/div/div[1]/div[1]/span/input'
    
    for item in array_items:
        driver.find_element(By.XPATH, sku_input_xpath).clear()
        driver.find_element(By.XPATH, sku_input_xpath).send_keys(item)
        driver.find_element(By.XPATH, sku_input_xpath).send_keys(Keys.ENTER)

def embed_size_check():
    embed_element = driver.find_element(By.XPATH, "/html/body/div[1]/div[2]/div[2]/div/div[2]/div[2]/div/embed")
    pdf_src = driver.find_element(By.XPATH, "/html/body/div[1]/div[2]/div[2]/div/div[2]/div[2]/div/embed").get_attribute('src')
    proc = re.search("(?<=,).*", pdf_src)
    base64_pdf_data = proc.group(0)
    bin_pdf_data = base64.b64decode(base64_pdf_data) #แปลง base64 to binary data
    
    embed_size = embed_element.size
    print(embed_size)
    print(pdf_src)
    print(f"base64 pdf extracted: {base64_pdf_data}")
    try:
        # with open("output.pdf", "wb") as pdf_file:
        #     pdf_file.write(bin_pdf_data)
        #     os.startfile("output.pdf", "print")
        print("printing")
    except OSError as err:
        print(f"No PDF Reader found. {err}")
    win32api.ShellExecute(
        0,
        "print",
        "output.pdf",
        '/d:"%s"' % win32print.GetDefaultPrinter(),
        ".",
        0
    )

driver = setup_chrome()
get_tabs()
driver.switch_to.window(merged_dict['SMCO :: พิมพ์ใบเสร็จซ้ำ'])
#* เอา function ที่ต้องการเทสมาใส่ข้างล่างนี่

#* function1
# item = ["CO6-010714", "CO6-010334"]
# fill_items(item)

#* function2
embed_size_check()





รายงานจำนวนtabs
counter[item] คือไร:  2
มี tabs ไรบ้าง {'SMCO :: พิมพ์ใบเสร็จซ้ำ': 'F406F341653AD970FF3F0451543470E9', 'Seller Centre': '6CD43438B110D8CAE362941D05742742', 'SMCO :: ลูกค้า': '47BA995F31C8E6463D86C19871695679', 'SMCO :: เปิดการขาย': 'B085C55120589895C8A2AF56A7410CED', 'SMCO :: เปิดการขาย1': 'E5EFDACBB2215ADA4BF16BBF29426DAC', 'SMCO :: ประวัติการขาย': '50A6179C34F206301D97E5320986A62D'}
{'height': 600, 'width': 1415}
...#toolbar=0
base64 pdf extracted: ...#toolbar=0
printing


### PDF READER

In [2]:
from pypdf import PdfReader
import pandas as pd
import re
from openpyxl import load_workbook
import os

extracted_txt:str =""
target_dir = r"TRB018324090900047-Tranfer.pdf"
reader = PdfReader(target_dir)
#* โหลดไฟล์ Excel ที่มีอยู่แล้ว
output_excel = r"Accel_mode.xlsx"

#* สกัดเอา ข้อความออกมาจากไฟล์
for page in reader.pages:
    extracted_txt += page.extract_text()
    
pattern = r'^.*?Product Code Barcode Product Name Transfer No\. Order Ship Status'
extracted_txt = re.sub(pattern, '', extracted_txt, flags=re.DOTALL)
extracted_txt = extracted_txt.lstrip()

pattern2 = r'ผู้ส่งสินค้า.*?(?:No\. Product Code Barcode Product Name Transfer No\. Order Ship Status|วันที่ _ _ _ / _ _ _ / _ _ _)'
extracted_txt = re.sub(pattern2, '', extracted_txt, flags=re.DOTALL)

pattern_serial = r'Serial\s:'
extracted_txt = re.sub(pattern_serial, '', extracted_txt, flags=re.DOTALL)

pattern_sku_no = r'\d+\s{0,}(?=([A-Z0-9]{3}-[0-9]{6}))'
extracted_txt = re.sub(pattern_sku_no, '', extracted_txt, flags=re.DOTALL)

print("อ่านค่าจาก", target_dir)
print(extracted_txt)


#* สกัดเอาค่าที่จำเป็นออกจากข้อความทั้งหมด
#* Regular expression สำหรับการจับ SKU
sku_pattern = r'([A-Z0-9]{3}-[0-9]{6})'

# *Regular expression สำหรับการจับ serial numbers
# serial_pattern = r'Shipped\s*([\w, \n]+)(?=(?:[A-Z0-9]{3}-[0-9]{6}|\nผู้ส่งสินค้า|$))'
serial_pattern = r'(?:Shipped|Confirm)\s*([\w, \n]+)(?=(?:[A-Z0-9]{3}-[0-9]{6}|\nผู้ส่งสินค้า|$))'


#* สกัด SKU
product_codes = re.findall(sku_pattern, extracted_txt)

#* สกัด serial numbers
serial_numbers = re.findall(serial_pattern, extracted_txt, re.DOTALL)

print(serial_numbers)

cleaned_serial_numbers = []
for serial in serial_numbers:
    #* ลบช่องว่างและเลขลำดับที่ไม่ต้องการออก
    cleaned_serial = re.sub(r'\n', '', serial).strip()  #* ลบเลขลำดับที่ท้าย
    # cleaned_serial = re.sub(r'\s+', '', cleaned_serial)  #* ลบช่องว่างทั้งหมด
    cleaned_serial_numbers.append(cleaned_serial)

#* แสดงผล
print("Product Codes:")
code_count = 0
for code in product_codes:
    code_count+=1
    print(code_count, " ", code)

print("\nSerial Numbers:")
code_count = 0
for serial in cleaned_serial_numbers:
    code_count+=1
    #* ลบช่องว่างและเพิ่มวงเล็บ [] รอบ Serial Numbers
    serial = serial.replace(" ", "")
    serial_list = serial.split(",")
    print(f"{code_count} {len(serial_list)} [{serial}]")



#* จัดการ serial numbers ให้เป็น list ของแต่ละ SKU
# serial_numbers_grouped = [serial.strip().replace('\n', '').replace(' ', '').split(',') for serial in serial_numbers]
serial_numbers_grouped = [re.findall(r'\b[\w]+\b', serial) for serial in cleaned_serial_numbers]

# ตรวจสอบข้อมูลที่ถูกสกัด
print("SKU Matches:")
print(len(product_codes),product_codes)
print("Serial Numbers Grouped:")
print(len(serial_numbers_grouped), serial_numbers_grouped)

#* สร้าง DataFrame ที่แต่ละคอลัมน์เป็น SKU และแต่ละ row เป็น serial number
data = {sku: serials for sku, serials in zip(product_codes, serial_numbers_grouped)}

# ตรวจสอบ DataFrame ก่อนเขียนลงไฟล์
print("DataFrame:")


#* เอาเข้าตาราง
try:
    # โหลด workbook และ sheet ล่าสุด
    book = load_workbook(output_excel)
    sheet = book.active

    # หาคอลัมน์ล่าสุดที่มีข้อมูล
    last_column = sheet.max_column
    
    # เขียนข้อมูลลงใน Excel
    for col, (sku, serials) in enumerate(data.items(), start=last_column+1):
        sheet.cell(row=1, column=col, value=sku)
        for row, serial in enumerate(serials, start=2):
            sheet.cell(row=row, column=col, value=serial)

    # บันทึกไฟล์
    book.save(output_excel)
    print(f"ข้อมูลถูกเพิ่มลงใน {output_excel} เรียบร้อยแล้ว")
except Exception as e:
    print(f"เกิดข้อผิดพลาด: {e}")
    import traceback
    traceback.print_exc()

อ่านค่าจาก TRB018324090900047-Tranfer.pdf
MNL-001561 DELL LED Monitor SE2422H -
23.8"/VA/75Hz/3Y1 1Shipped
 
 
9XQRB14
MNL-001633 AOC  Curved Gaming Monitor 27"
C27G2Z/67 VA/240Hz/0.5ms/FHD2 2Shipped
 
 
XFXP7JA002517, XFXP7JA002516
MNL-001743 DELL LED Monitor U2723QE -
27"/4K/IPS/60Hz/3Y*33 3Shipped
 
 
2QKKL04, 2NKKL04, HNKKL04
MNL-001850 COOLER MASTER Gaming Monitor 27"
GA271 VA/100Hz/1ms/2K WQHD3 3Shipped
 
 
CMIGA271US1241800002, CMIGA271US1241800021, CMIGA271US1241800042
MNL-001856 MSI Curved Gaming Monitor G27C4X -
27"/VA/250Hz/1m/FreeSync2 2Shipped
 
 
CA9T914603509, CA9T914603388
MNL-001886 DAHUA Gaming Monitor Curved LM30-
E330C - 30"/VA/200Hz/3Y*35 5Shipped
 
 
AFU0100370EZ00468, AE0EBDAPA100013, AE0EBDAPA100020, AE0EBDAPA100017, AE0EBDAPA100004
MNL-001887 DAHUA Gaming Monitor Curved LM34-
E330C - 34"/VA/165Hz/3Y*32 2Shipped
 
 
AD40100370KE00081, AD40100370KE00312
MNL-001941 HP Gaming Monitor OMEN 24 -
23.8"/IPS/165Hz/3Y5 5Shipped
 
 
CNC40410N4, CNC41535W8, CNC42428GT, CNC

In [33]:
from tkinter import filedialog




def sn_extractor(output_excel, target_dir):
    extracted_txt:str = ""
    # target_dir = r"C:\Users\ONLINE_MIS\Downloads\TRB018324080900002-Tranfer.pdf" //example
    # target_dir = target_dir
    reader = PdfReader(target_dir)
    #* โหลดไฟล์ Excel ที่มีอยู่แล้ว
    # output_excel = r"C:\Users\ONLINE_MIS\Downloads\Accel_mode.xlsx" //example
    # output_excel = output_excel

    #* สกัดเอา ข้อความออกมาจากไฟล์
    for page in reader.pages:
        extracted_txt += page.extract_text()
        
    pattern = r'^.*?(?=No\. Product Code Barcode Product Name Transfer No\. Order Ship Status)'
    extracted_txt = re.sub(pattern, '', extracted_txt, flags=re.DOTALL)
    extracted_txt = extracted_txt.lstrip()
    
    # print(extracted_txt)
    
    #* สกัดเอาค่าที่จำเป็นออกจากข้อความทั้งหมด
    #* Regular expression สำหรับการจับ SKU
    sku_pattern = r'([A-Z0-9]{3}-[0-9]{6})'

    # *Regular expression สำหรับการจับ serial numbers
    serial_pattern = r'Shipped\s+([\w,\s]+)(?=Serial\s*:)'

    #* สกัด SKU
    sku_matches = re.findall(sku_pattern, extracted_txt)

    #* สกัด serial numbers
    serial_matches = re.findall(serial_pattern, extracted_txt, re.DOTALL)

    #* จัดการ serial numbers ให้เป็น list ของแต่ละ SKU
    # serial_numbers_grouped = [serial.strip().replace('\n', '').replace(' ', '').split(',') for serial in serial_matches]
    serial_numbers_grouped = [re.findall(r'\b[\w]+\b', serial) for serial in serial_matches]

    # ตรวจสอบข้อมูลที่ถูกสกัด
    print("SKU Matches:")
    print(len(sku_matches),sku_matches)
    print("Serial Numbers Grouped:")
    print(len(serial_numbers_grouped), serial_numbers_grouped)

    #* สร้าง DataFrame ที่แต่ละคอลัมน์เป็น SKU และแต่ละ row เป็น serial number
    data = {sku: serials for sku, serials in zip(sku_matches, serial_numbers_grouped)}

    # ตรวจสอบ DataFrame ก่อนเขียนลงไฟล์
    print("DataFrame:")


    #* เอาเข้าตาราง
    try:
        # โหลด workbook และ sheet ล่าสุด
        book = load_workbook(output_excel)
        sheet = book.active

        # หาคอลัมน์ล่าสุดที่มีข้อมูล
        last_column = sheet.max_column
        
        # เขียนข้อมูลลงใน Excel
        for col, (sku, serials) in enumerate(data.items(), start=last_column+1):
            sheet.cell(row=1, column=col, value=sku)
            for row, serial in enumerate(serials, start=2):
                sheet.cell(row=row, column=col, value=serial)

        # บันทึกไฟล์
        book.save(output_excel)
        print(f"ข้อมูลถูกเพิ่มลงใน {output_excel} เรียบร้อยแล้ว")
    except Exception as e:
        print(f"เกิดข้อผิดพลาด: {e}")
        import traceback
        traceback.print_exc()

def extract_sn_btn(accel_file_dir):
    if not accel_file_dir:
        print("select accel file first!!")
        return
    
    target_dirs:tuple = filedialog.askopenfilenames()
    if len(target_dirs) != 0:
        for target_dir in target_dirs:
            sn_extractor(accel_file_dir, target_dir)
    else:
        print("You have not selected any transfer file, Extraction ends!!")

# Test Loguru


In [17]:
from loguru import logger
import threading
orders = ["order1", "order2", "order3", "order4", "order5", "order6", "order7", "order8", "order9", "order10"]



def operation_start(order):
    logger.add("autopageMKII_jupyter_log.log", format="{time} {level} {message}", level="INFO")
    logger.info(f"{order}  Start!!")
    logger.info(f"{order}  Stop!!")
    


for order in orders:
    test_thread = threading.Thread(target=lambda: operation_start(order), name="x")
    
    test_thread.start()
    print(test_thread.is_alive(), "B4 join")
    test_thread.join()
    print(test_thread.is_alive(), "After join")


print(test_thread.name)

2024-08-23 14:59:44.702 | INFO     | __main__:operation_start:9 - order1  Start!!
2024-08-23 14:59:44.719 | INFO     | __main__:operation_start:10 - order1  Stop!!
2024-08-23 14:59:44.722 | INFO     | __main__:operation_start:9 - order2  Start!!
2024-08-23 14:59:44.733 | INFO     | __main__:operation_start:10 - order2  Stop!!
2024-08-23 14:59:44.733 | INFO     | __main__:operation_start:9 - order3  Start!!
2024-08-23 14:59:44.749 | INFO     | __main__:operation_start:10 - order3  Stop!!
2024-08-23 14:59:44.755 | INFO     | __main__:operation_start:9 - order4  Start!!
2024-08-23 14:59:44.760 | INFO     | __main__:operation_start:10 - order4  Stop!!
2024-08-23 14:59:44.765 | INFO     | __main__:operation_start:9 - order5  Start!!
2024-08-23 14:59:44.765 | INFO     | __main__:operation_start:10 - order5  Stop!!
2024-08-23 14:59:44.780 | INFO     | __main__:operation_start:9 - order6  Start!!
2024-08-23 14:59:44.780 | INFO     | __main__:operation_start:10 - order6  Stop!!
2024-08-23 14:59

True B4 join
False After join
True B4 join
False After join
True B4 join
False After join
True B4 join
False After join
True B4 join
False After join
True B4 join
False After join
True B4 join
False After join
True B4 join
False After join
True B4 join
False After join
True B4 join
False After join
x


---
### Thread

In [4]:
import threading
import time
import tkinter as tk

# Global variables
current_thread = None
stop_event = threading.Event()

def thread_function():
    while not stop_event.is_set():
        print("Thread is running...")
        time.sleep(1)

def start_thread():
    global current_thread, stop_event

    # Set the event to signal any existing thread to stop
    if current_thread and current_thread.is_alive():
        stop_event.set()

        # Use after() to wait for the existing thread to actually stop
        def wait_for_stop():
            if current_thread.is_alive():
                root.after(100, wait_for_stop)  # Check again after 100ms
            else:
                # Reset the event for the new thread
                stop_event.clear()
                # Start a new thread
                current_thread = threading.Thread(target=thread_function)
                current_thread.start()

        # Start waiting for the current thread to stop
        wait_for_stop()
    else:
        # Reset the event for the new thread
        stop_event.clear()
        # Start a new thread if no thread is running
        current_thread = threading.Thread(target=thread_function)
        current_thread.start()

# Setup the GUI
root = tk.Tk()
root.title("Thread Control")

start_button = tk.Button(root, text="Start New Thread", command=start_thread)
start_button.pack(pady=20)

root.mainloop()



Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.1520.0_x64__qbz5n2kfra8p0\Lib\tkinter\__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\BCP_27\AppData\Local\Temp\ipykernel_12352\3688568310.py", line 18, in start_thread
    if current_thread and current_thread.is_alive():
       ^^^^^^^^^^^^^^
UnboundLocalError: cannot access local variable 'current_thread' where it is not associated with a value


---
# Price Pattern Memorizer


### Pattern

In [ ]:
#* เราจะสร้าง cache เก็บไว้ อันนึง สำหรับเก็บค่าทั้งหมด ที่มี Pattern
cache = {}

#todo find_elements(BY.css_selector, 'div.col-sm-12.panel.panel-default.ng-scope') จะเปนการเลือกที่ element รายการ item โดยตรง ใช้เป็น find_elements เราจะได้ array รายการสินค้ามา เราก็ตัด อันแรกออก เพราะมันเปนค่าขนส่ง
#? nth element of its type  
#? nth ในที่นี้ย่อมาจากคำว่า "n-th" ซึ่งเป็นรูปแบบทั่วไปของการระบุลำดับในคณิตศาสตร์และโปรแกรมมิ่ง เช่น 4th 5th มั้ง
#? ฉะนั้น nth element of its type  หมายถึง ลำดับ element ที่เปน type เดียวกัน, ฉะนั้น(อีกแล้ว)div:nth-of-type(2)ก็จะหมายถึง div ตัวที่2, ถ้ามี a ปนใน div ก็ใช้ a:nth-of-type แทนไงล่ะ
#todo element location ของค่าต่างๆ css_selector สำหรับแต่ละ item, ต่อli
{
    "sku":"div.col-sm-12.panel.panel-default.ng-scope div div div:nth-of-type(2) a",
    "srp":"div.col-sm-6 div.row div a.col-sm-6.text-right.font-color-base.ng-binding"
    
 }

from datetime import datetime
date_str = "2024-11-11 00:01"
date_value = datetime.strptime(date_str, "%Y-%m-%d %H:%M")
x = datetime.now()

time_diff = x - date_value

print("ความต่างของเวลา: ", time_diff)


x = {}

x["manual_discount"] = 100
x["cp"] = ""
x["ordered_time"] = "2024-11-11 00:01"

cache["CO6-000001"] = x
print(cache)
